# Demo 5 — The three-layer caching stack — and the false hit

**AI Cost Management and Token Utilization** · Module 2 · ~8 minutes

> Runs top-to-bottom on live API keys. Every cell that spends money prints what it spent.

---

## What this demo lands

1. Exact, semantic, and provider prompt caching are **three different things** that stack.
2. Semantic caching skips the call entirely — 100% saving on a hit.
3. **A hit rate without a false-hit rate is a savings claim with the risk removed from the page.**

In [ ]:
# --- Setup: install + keys -------------------------------------------------
# Colab: this cell installs everything. Local: it is a no-op if already installed.
%pip install -q anthropic openai tiktoken pandas matplotlib 2>/dev/null

import os, getpass

def need(var):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f'{var}: ')
    return os.environ[var]

# You need at least one. Anthropic is used for the cache-metadata demos because
# it reports cache reads in a separate, easily inspectable bucket.
need('ANTHROPIC_API_KEY')
# need('OPENAI_API_KEY')   # uncomment if you want the OpenAI comparisons
print('keys loaded')

In [ ]:
# --- Verified rate card, September 2026 ------------------------------------
# Sources (checked 5 Sept 2026):
#   platform.claude.com/docs/en/about-claude/pricing
#   developers.openai.com/api/docs/pricing
#   ai.google.dev/gemini-api/docs/pricing
#   deepseek.ai/pricing
# USD per 1,000,000 tokens.  cache_w = 5-minute cache write, cache_r = cache read.

PRICES = {
    # model id                       input  output  cache_w  cache_r
    'deepseek-v4-flash':            dict(inp=0.14, out=0.28,  cw=0.14,  cr=0.0028),
    'gpt-5.6-luna':                 dict(inp=0.20, out=1.20,  cw=0.25,  cr=0.02),
    'gemini-3.5-flash-lite':        dict(inp=0.30, out=2.50,  cw=0.30,  cr=0.03),
    'gemini-3.8-flash':             dict(inp=0.75, out=3.75,  cw=0.75,  cr=0.075),
    'claude-haiku-4-5':             dict(inp=1.00, out=5.00,  cw=1.25,  cr=0.10),
    'claude-sonnet-5':              dict(inp=2.00, out=10.00, cw=2.50,  cr=0.20),
    'gpt-5.6-terra':                dict(inp=2.00, out=12.00, cw=2.50,  cr=0.20),
    'claude-opus-5':                dict(inp=5.00, out=25.00, cw=6.25,  cr=0.50),
    'gpt-5.6-sol':                  dict(inp=4.00, out=20.00, cw=5.00,  cr=0.40),
    'gpt-6-astra':                  dict(inp=10.00,out=50.00, cw=12.50, cr=1.00),
    'claude-fable-5-1':             dict(inp=10.00,out=50.00, cw=12.50, cr=0.25),
}

def cost(model, inp=0, out=0, cache_w=0, cache_r=0):
    """Cost in USD for one call, given token counts by billing bucket."""
    p = PRICES[model]
    return (inp*p['inp'] + out*p['out'] + cache_w*p['cw'] + cache_r*p['cr']) / 1e6

def usd(x):
    return f'${x:,.6f}' if x < 0.01 else f'${x:,.4f}' if x < 1 else f'${x:,.2f}'

print(f'{len(PRICES)} models loaded')

In [ ]:
# --- Cost ledger: every billable call in this notebook lands here ----------
import pandas as pd
LEDGER = []

def log_call(label, model, inp=0, out=0, cache_w=0, cache_r=0, note=''):
    c = cost(model, inp, out, cache_w, cache_r)
    LEDGER.append(dict(label=label, model=model, input=inp, output=out,
                       cache_write=cache_w, cache_read=cache_r, usd=c, note=note))
    print(f'{label:<38} {usd(c):>12}   in={inp:<7} out={out:<6} cw={cache_w:<7} cr={cache_r:<7} {note}')
    return c

def ledger():
    df = pd.DataFrame(LEDGER)
    if df.empty:
        print('no calls yet'); return df
    print(f'\nTOTAL SPENT IN THIS NOTEBOOK: {usd(df.usd.sum())}')
    return df

In [ ]:
%pip install -q numpy anthropic 2>/dev/null
import numpy as np, time, hashlib, math
from collections import Counter
import anthropic
client = anthropic.Anthropic()
MODEL = 'claude-sonnet-5'

---
## 1. The stack

```
[L1] Exact match     <1ms     byte-identical query        skips the call
  v miss
[L2] Semantic cache  3-10ms   cosine >= threshold         skips the call
  v miss
[L3] Prompt cache    50-200ms identical prefix            90-98% off INPUT only
  v miss
[L4] Model call      0.5-2s   full price
```

In [ ]:
# Stopwords + light stemming so this toy embedding behaves like a real one for the demo.
STOP = {'the','is','a','an','what','how','do','does','did','can','could','you','your',
        'me','my','i','about','tell','please','of','to','for','and','it','in','on','are'}

def embed(text):
    """Local bag-of-words embedding, stopwords removed and plurals stemmed.
    Swap for a real embedding model in production — the cache mechanics and the
    failure modes below are identical either way; only the threshold changes."""
    words = []
    for w in text.lower().replace('?','').replace('.','').replace(',','').split():
        w = w.rstrip('s') if len(w) > 4 and w.endswith('s') else w
        if w not in STOP and len(w) > 2:
            words.append(w)
    return Counter(words)

def cos(a, b):
    common = set(a) & set(b)
    num = sum(a[w]*b[w] for w in common)
    den = math.sqrt(sum(v*v for v in a.values())) * math.sqrt(sum(v*v for v in b.values()))
    return num/den if den else 0.0

class CacheStack:
    def __init__(self, threshold=0.85):
        self.exact, self.semantic, self.threshold = {}, [], threshold
        self.stats = Counter()

    def get(self, q):
        key = hashlib.sha256(q.encode()).hexdigest()
        if key in self.exact:
            self.stats['L1_exact'] += 1
            return self.exact[key], 'L1_exact', 1.0
        qv = embed(q)
        best, best_s = None, 0.0
        for cv, cq, ans in self.semantic:
            s = cos(qv, cv)
            if s > best_s: best, best_s = (cq, ans), s
        if best and best_s >= self.threshold:
            self.stats['L2_semantic'] += 1
            return best[1], 'L2_semantic', best_s
        self.stats['L4_miss'] += 1
        return None, 'L4_miss', best_s

    def put(self, q, ans):
        self.exact[hashlib.sha256(q.encode()).hexdigest()] = ans
        self.semantic.append((embed(q), q, ans))

cache = CacheStack(threshold=0.85)
print('cache ready')

---
## 2. Run realistic traffic — with the paraphrases every support queue has

In [ ]:
SYSTEM = 'You are a concise support assistant for Northwind Logistics. Answer in one sentence.'

TRAFFIC = [
  'What is your refund policy?',
  'What is your refund policy?',                 # exact repeat
  'Can you tell me about the refund policy?',    # paraphrase
  'How do refunds work here?',                   # paraphrase
  'How do I track my shipment?',
  'How can I track my shipment?',                # paraphrase
  'What are your delivery hours?',
  'What is your refund policy?',                 # exact repeat again
  'Tell me how to track a shipment',             # paraphrase
  'Do you ship internationally?',
]

spent = 0.0
for q in TRAFFIC:
    ans, layer, score = cache.get(q)
    if ans is None:
        r = client.messages.create(model=MODEL, max_tokens=90, system=SYSTEM,
                                   messages=[{'role':'user','content':q}])
        ans = r.content[0].text
        spent += log_call(f'MISS  {q[:34]}', MODEL,
                          inp=r.usage.input_tokens, out=r.usage.output_tokens)
        cache.put(q, ans)
    else:
        print(f'{layer:<12} {q[:34]:<36} sim={score:.2f}   $0.000000  (call skipped)')

hits = cache.stats['L1_exact'] + cache.stats['L2_semantic']
print(f'\nhit rate      {hits}/{len(TRAFFIC)} = {hits/len(TRAFFIC):.0%}')
print(f'spent         {usd(spent)}')
print(f'without cache {usd(spent * len(TRAFFIC) / max(1, cache.stats["L4_miss"]))} (approx)')
print(f'saving        {1 - cache.stats["L4_miss"]/len(TRAFFIC):.0%} of all calls eliminated')

---
## 3. THE FALSE HIT — the demo nobody runs

Two queries with very high similarity and **opposite** correct answers.
One of the actions is irreversible.

In [ ]:
q_a = 'Please cancel the order.'
q_b = 'Please cancel the standing order.'

sim = cos(embed(q_a), embed(q_b))
print(f'cosine similarity: {sim:.3f}')
print(f'threshold:         {cache.threshold}')
print(f'-> would be served from cache: {sim >= cache.threshold}')
print()
print('One cancels a single shipment. The other cancels a recurring contract.')
print('The cache cannot tell. It has no concept of what the action costs.')

### The controls

In [ ]:
RISK_TIERS = {
  'informational': dict(threshold=0.82, cacheable=True,  ttl_s=86400),
  'account':       dict(threshold=0.95, cacheable=True,  ttl_s=300),
  'transactional': dict(threshold=1.00, cacheable=False, ttl_s=0),
  'legal_medical': dict(threshold=1.00, cacheable=False, ttl_s=0),
}
for tier, cfg in RISK_TIERS.items():
    print(f'{tier:<16} threshold={cfg["threshold"]:<6} cacheable={str(cfg["cacheable"]):<6} ttl={cfg["ttl_s"]}s')
print()
print('Plus: namespace the cache per tenant, per user, and per entitlement level.')
print('A cache that crosses a permission boundary is a data incident, not a cost win.')

### Measure the pair, always

In [ ]:
def report(hits, total, false_hits, sampled):
    print(f'hit rate        {hits/total:.1%}')
    print(f'false-hit rate  {false_hits/max(1,sampled):.1%}  (from {sampled} human-reviewed samples)')
    print(f'net saving      {(hits-false_hits)/total:.1%}')
    if false_hits/max(1,sampled) > 0.02:
        print('\nWARNING: raise the threshold. You are buying savings with wrong answers.')

report(hits=cache.stats['L1_exact']+cache.stats['L2_semantic'],
       total=len(TRAFFIC), false_hits=0, sampled=20)
print('\nYou can drive hit rate to 90% by lowering the threshold. That is not a win.')

In [ ]:
ledger()

---
## Takeaways

- L1 and L2 **skip the call**. L3 only discounts the input. They are complements, not alternatives.
- Set the threshold **by risk class**, not globally.
- Namespace by tenant, user and entitlement. TTL by how fast the underlying truth changes.
- **Report hit rate and false-hit rate on the same dashboard, always.**